# Statistical Foundations for Network Anomaly Detection

This notebook demonstrates statistical methods for detecting anomalies in network traffic,
progressing from simple baseline modeling to SVM-based classification.

## Contents

1. **Statistical Baseline Modeling** - Mean, standard deviation, z-scores
2. **Hypothesis Testing** - Formalizing anomaly detection as statistical inference
3. **Time Series Preparation** - Temporal structure, rolling statistics, stationarity
4. **CUSUM Change Detection** - Sequential analysis for detecting shifts
5. **SVM Classification** - Supervised learning for traffic classification
6. **Threshold Calibration** - Balancing detection rate vs analyst workload

In [ ]:
import sys
from pathlib import Path

# Make scripts/ importable
sys.path.insert(0, str(Path.cwd().parent))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# Project imports
from scripts.zeek_to_dataframe import load_zeek_log, CONN_SCHEMA, DNS_SCHEMA
from scripts.normalize import normalize_conn, normalize_dns, merge_normalized

# Display settings
pd.set_option("display.max_columns", None)
pd.set_option("display.width", 160)
%matplotlib inline

# Set random seed for reproducibility
np.random.seed(42)

In [ ]:
# Load and normalize sample data
ZEEK_DIR = Path("../data/zeek_logs/sample")

conn_raw = load_zeek_log(ZEEK_DIR / "conn.log", schema=CONN_SCHEMA)
dns_raw = load_zeek_log(ZEEK_DIR / "dns.log", schema=DNS_SCHEMA)

conn = normalize_conn(conn_raw)
dns = normalize_dns(dns_raw)

print(f"Loaded {len(conn)} connections and {len(dns)} DNS queries")
conn.head()

---
# Part 1: Statistical Baseline Modeling

## Why Baseline Modeling Before Machine Learning?

Before applying ML algorithms, we must understand normal behavior through statistical baselines:

1. **Interpretability**: Z-scores are immediately actionable ("4 std from mean")
2. **Computational Efficiency**: O(n) computation, runs on edge devices
3. **Feature Engineering**: Reveals which metrics have useful variance
4. **Threshold Calibration**: Grounds ML scores in statistical theory
5. **Drift Detection**: Baselines reveal when retraining is needed

In [ ]:
from scripts.baseline_modeling import BaselineModel, compute_per_host_baselines

# Create and fit baseline model
model = BaselineModel(z_threshold=2.0)
model.fit(conn, dns, time_window="5s")

print(model.get_baseline_report())

In [ ]:
# Detect anomalies using fitted baselines
anomalies = model.detect_anomalies(conn, dns, time_window="5s")

print("\nAll observations with z-scores:")
print(anomalies.head(10))

print("\nFlagged anomalies (|z| > 2.0):")
print(anomalies[anomalies["is_anomaly"]])

In [ ]:
# Per-host baseline analysis
host_baselines = compute_per_host_baselines(conn, dns, host_column="src_ip")
print("\nPer-host baselines:")
host_baselines

---
# Part 2: Hypothesis Testing

Formalizing anomaly detection as statistical inference:

- **H₀ (Null Hypothesis)**: Traffic behavior is NORMAL
- **H₁ (Alternative)**: Traffic behavior is ANOMALOUS

This provides:
- **Quantified confidence** via p-values
- **Controlled false positive rate** via significance level α
- **Sample-size awareness** (detecting anomaly in 10 vs 10,000 observations)

In [ ]:
from scripts.hypothesis_testing import (
    AlertingFramework, 
    test_failed_connection_rate,
    test_nxdomain_rate,
    proportion_test,
)

# Single test: Is the failure rate anomalous?
result = test_failed_connection_rate(
    conn,
    baseline_rate=0.05,  # Assume 5% baseline
    alpha=0.05
)

print(f"Test: {result.test_name}")
print(f"Observed: {result.observed_value:.4f}")
print(f"Expected: {result.expected_value:.4f}")
print(f"z-statistic: {result.test_statistic:.2f}")
print(f"p-value: {result.p_value:.6f}")
print(f"Reject H₀: {result.reject_null}")
print(f"Severity: {result.severity.value}")
print(f"\n{result.interpretation}")

In [ ]:
# Complete alerting framework
framework = AlertingFramework(
    baseline_failed_rate=0.05,
    baseline_nxdomain_rate=0.10,
    alpha=0.05
)

print(framework.generate_report(conn, dns))

### Mapping p-values to Operational Alerts

| p-value | Severity | Action |
|---------|----------|--------|
| < 0.001 | CRITICAL | Investigate immediately |
| < 0.01  | HIGH     | Investigate within 1 hour |
| < 0.05  | MEDIUM   | Daily review queue |
| ≥ 0.05  | LOW      | Log for trends only |

---
# Part 3: Time Series Preparation

## Key Concepts

### Nonstationarity
Network traffic is typically **non-stationary**: its statistical properties change over time
(business hours vs nights, week vs weekend, before/after deployments).

### Seasonality
Traffic has **seasonal patterns**: hourly (cron jobs), daily (business hours), weekly (Mondays).

### Why Random Train/Test Split is Invalid
For time series:
- **Temporal leakage**: Training data shouldn't contain future information
- **Autocorrelation**: Adjacent points are correlated
- Must use **temporal split**: train on past, test on future

In [ ]:
from scripts.timeseries_prep import (
    TimeSeriesPreprocessor,
    temporal_train_test_split,
    generate_timeseries_report,
)

# Transform to time series with 5-second buckets
prep = TimeSeriesPreprocessor(bucket_size="5s", window_size=3)
ts_data = prep.transform(conn, dns)

print(f"Time series shape: {ts_data.shape}")
print(f"\nColumns: {list(ts_data.columns)}")
ts_data.head(10)

In [ ]:
# Rolling statistics show how metrics evolve over time
fig, axes = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Plot failed rate with rolling mean
ax1 = axes[0]
ax1.plot(ts_data.index, ts_data["failed_rate"], "b-", alpha=0.5, label="Raw")
if "failed_rate_rolling_mean" in ts_data.columns:
    ax1.plot(ts_data.index, ts_data["failed_rate_rolling_mean"], "r-", linewidth=2, label="Rolling Mean")
ax1.set_ylabel("Failed Rate (%)")
ax1.legend()
ax1.grid(True, alpha=0.3)

# Plot rolling z-score
ax2 = axes[1]
if "failed_rate_rolling_zscore" in ts_data.columns:
    ax2.plot(ts_data.index, ts_data["failed_rate_rolling_zscore"], "g-")
    ax2.axhline(y=2, color="r", linestyle="--", label="z=2")
    ax2.axhline(y=-2, color="r", linestyle="--")
ax2.set_ylabel("Rolling Z-Score")
ax2.set_xlabel("Time")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Temporal train/test split (NOT random!)
train, val, test = temporal_train_test_split(ts_data, train_ratio=0.7, val_ratio=0.15)

print(f"Temporal Split:")
print(f"  Train: {len(train)} buckets ({train.index.min()} to {train.index.max()})")
print(f"  Val:   {len(val)} buckets")
print(f"  Test:  {len(test)} buckets ({test.index.min()} to {test.index.max()})")
print("\nNote: Each split contains contiguous time periods (no random mixing)")

---
# Part 4: CUSUM Change Detection

CUSUM (Cumulative Sum) detects **when** the mean of a process shifts:

$$S_n^+ = \max(0, S_{n-1}^+ + (x_n - \mu) - k)$$

Where:
- $x_n$ = current observation
- $\mu$ = target mean (baseline)
- $k$ = allowance (slack)
- $h$ = threshold for change detection

**Why CUSUM for security?**
- Early detection of persistent shifts (attacks)
- Low false positive rate (random noise cancels out)
- Pinpoints WHEN the change occurred

In [ ]:
from scripts.cusum_detector import (
    CUSUMDetector,
    detect_connection_failure_changes,
    generate_cusum_report,
    plot_cusum,
)

# Create synthetic data with a change point for demonstration
np.random.seed(42)
n_before = 30
n_after = 30

# Simulate: Normal (5% failure) -> Attack (25% failure)
baseline_rates = np.random.normal(5, 2, n_before).clip(0, 100)
attack_rates = np.random.normal(25, 5, n_after).clip(0, 100)
synthetic_rates = np.concatenate([baseline_rates, attack_rates])

# Create timestamps
timestamps = pd.date_range("2024-01-01 08:00", periods=len(synthetic_rates), freq="1min")
synthetic_series = pd.Series(synthetic_rates, index=timestamps)

print(f"Synthetic data: {len(synthetic_series)} observations")
print(f"Change point at observation {n_before} (minute 30)")

In [ ]:
# Fit CUSUM detector on baseline period, then detect changes
detector = CUSUMDetector()
detector.fit_from_data(synthetic_series.iloc[:20], k_factor=0.5, h_factor=4.0)

print(f"Detector parameters:")
print(f"  Target mean: {detector.target_mean:.2f}")
print(f"  Std dev:     {detector.std:.2f}")
print(f"  k (slack):   {detector.k:.2f}")
print(f"  h (threshold): {detector.h:.2f}")

# Run detection
results = detector.detect_all(synthetic_series, timestamps=timestamps)

print(f"\nDetected {len(detector.change_points)} change point(s)")

In [ ]:
# Visualize CUSUM detection
fig, (ax1, ax2) = plt.subplots(2, 1, figsize=(12, 6), sharex=True)

# Top: Original series with change point
ax1.plot(results["timestamp"], results["value"], "b-", alpha=0.7, label="Failed Rate")
ax1.axvline(x=timestamps[n_before], color="orange", linestyle="--", label="True Change")

# Mark detected changes
changes = results[results["change_detected"]]
if len(changes) > 0:
    ax1.scatter(changes["timestamp"], changes["value"], c="red", s=100, 
               marker="^", zorder=5, label="Detected")

ax1.set_ylabel("Failed Rate (%)")
ax1.legend()
ax1.grid(True, alpha=0.3)
ax1.set_title("CUSUM Change Detection")

# Bottom: CUSUM values
ax2.plot(results["timestamp"], results["upper_cusum"], "g-", label="Upper CUSUM")
ax2.plot(results["timestamp"], results["lower_cusum"], "r-", label="Lower CUSUM")
ax2.axhline(y=detector.h, color="purple", linestyle=":", label=f"Threshold (h={detector.h:.1f})")
ax2.set_xlabel("Time")
ax2.set_ylabel("Cumulative Sum")
ax2.legend()
ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
print(generate_cusum_report(detector, results, "Failed Connection Rate"))

---
# Part 5: SVM Classification

## Support Vector Machines for Anomaly Detection

SVMs find the optimal hyperplane that separates classes with **maximum margin**.

### Key Concepts

**The Role of C (Regularization)**:
- **Small C (0.1)**: Soft margin, tolerates misclassifications → better generalization
- **Large C (10)**: Hard margin, fits training data closely → may overfit

**The Role of Kernel**:
- **Linear**: Fast, good for high-dimensional data
- **RBF**: Flexible, handles non-linear boundaries

### Operational Trade-offs

| | False Positives | False Negatives |
|--|--|--|
| Impact | Analyst time wasted | Attacks missed |
| High tolerance | Mature SOC, limited analysts | Low-risk systems |
| Low tolerance | Automated response enabled | Critical assets |

In [ ]:
from scripts.svm_classifier import (
    SVMClassifier,
    prepare_features,
    generate_classification_report,
)
from sklearn.model_selection import train_test_split

# Prepare features from network data
X, y = prepare_features(conn, dns)

print(f"Feature matrix: {X.shape}")
print(f"Features: {list(X.columns)}")
print(f"\nLabel distribution:")
print(y.value_counts())

In [ ]:
# Split data (temporal split would be better for production)
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.3, random_state=42,
    stratify=y if y.sum() > 1 else None
)

print(f"Training set: {len(X_train)} samples")
print(f"Test set: {len(X_test)} samples")

In [ ]:
# Train SVM classifier
classifier = SVMClassifier(
    kernel="rbf",
    C=1.0,
    class_weight="balanced"  # Handle imbalanced classes
)
classifier.fit(X_train, y_train)

# Evaluate
print(generate_classification_report(classifier, X_test, y_test))

In [ ]:
# Compare different C values
print("Effect of C parameter:")
print("-" * 50)

for C in [0.1, 1.0, 10.0]:
    clf = SVMClassifier(kernel="rbf", C=C, class_weight="balanced")
    clf.fit(X_train, y_train)
    metrics = clf.evaluate(X_test, y_test)
    print(f"C={C:4.1f}: Precision={metrics.precision:.3f}, Recall={metrics.recall:.3f}, F1={metrics.f1:.3f}")

In [ ]:
# Compare kernels
print("\nEffect of kernel choice:")
print("-" * 50)

for kernel in ["linear", "rbf", "poly"]:
    clf = SVMClassifier(kernel=kernel, C=1.0, class_weight="balanced")
    clf.fit(X_train, y_train)
    metrics = clf.evaluate(X_test, y_test)
    print(f"{kernel:8s}: Precision={metrics.precision:.3f}, Recall={metrics.recall:.3f}, F1={metrics.f1:.3f}")

---
# Part 6: Threshold Calibration

## The Decision Threshold Dilemma

ML classifiers output probabilities; the **threshold** converts these to decisions:
- `score >= threshold` → Alert
- `score < threshold` → No alert

### Trade-off

| Low Threshold (0.2) | High Threshold (0.8) |
|---------------------|----------------------|
| Many alerts | Few alerts |
| High recall (catch more attacks) | Low recall (miss attacks) |
| High FP rate (analyst burden) | Low FP rate |
| For: Critical assets | For: Limited SOC capacity |

In [ ]:
from scripts.threshold_calibration import (
    ThresholdAnalyzer,
    analyze_workload_scenarios,
    generate_calibration_report,
)

# Get probability scores from classifier
y_scores = classifier.predict_proba(X_test)[:, 1]  # Probability of positive class

# Create analyzer
analyzer = ThresholdAnalyzer(y_test.values, y_scores)

print(f"ROC AUC: {analyzer.roc_auc:.4f}")

In [ ]:
# Find optimal threshold
optimal_f1 = analyzer.find_optimal_threshold(maximize="f1")
optimal_95recall = analyzer.find_optimal_threshold(target_recall=0.95)

print(f"Optimal for F1: threshold={optimal_f1.threshold:.3f}")
print(f"  Precision: {optimal_f1.precision:.3f}")
print(f"  Recall:    {optimal_f1.recall:.3f}")
print(f"  F1:        {optimal_f1.f1:.3f}")

print(f"\nFor 95% recall: threshold={optimal_95recall.threshold:.3f}")
print(f"  Precision: {optimal_95recall.precision:.3f}")
print(f"  Recall:    {optimal_95recall.recall:.3f}")
print(f"  Alert rate: {optimal_95recall.alert_rate*100:.1f}%")

In [ ]:
# Visualize threshold trade-offs
df = analyzer.get_threshold_df()

fig, axes = plt.subplots(2, 2, figsize=(12, 8))

# Precision-Recall curve
ax1 = axes[0, 0]
ax1.plot(analyzer.recall_curve, analyzer.precision_curve, "b-", linewidth=2)
ax1.fill_between(analyzer.recall_curve, analyzer.precision_curve, alpha=0.3)
ax1.set_xlabel("Recall")
ax1.set_ylabel("Precision")
ax1.set_title("Precision-Recall Curve")
ax1.grid(True, alpha=0.3)

# ROC curve
ax2 = axes[0, 1]
ax2.plot(analyzer.fpr_curve, analyzer.tpr_curve, "b-", linewidth=2,
        label=f"ROC (AUC={analyzer.roc_auc:.3f})")
ax2.plot([0, 1], [0, 1], "k--", alpha=0.5)
ax2.set_xlabel("False Positive Rate")
ax2.set_ylabel("True Positive Rate")
ax2.set_title("ROC Curve")
ax2.legend()
ax2.grid(True, alpha=0.3)

# Metrics vs Threshold
ax3 = axes[1, 0]
ax3.plot(df["threshold"], df["f1"], "b-", linewidth=2, label="F1")
ax3.plot(df["threshold"], df["precision"], "g--", label="Precision")
ax3.plot(df["threshold"], df["recall"], "r--", label="Recall")
ax3.axvline(x=optimal_f1.threshold, color="purple", linestyle=":", 
           label=f"Optimal ({optimal_f1.threshold:.2f})")
ax3.set_xlabel("Threshold")
ax3.set_ylabel("Score")
ax3.set_title("Metrics vs Threshold")
ax3.legend()
ax3.grid(True, alpha=0.3)

# Alert rate vs Threshold
ax4 = axes[1, 1]
ax4.plot(df["threshold"], df["alert_rate"] * 100, "b-", linewidth=2)
ax4.fill_between(df["threshold"], df["alert_rate"] * 100, alpha=0.3)
ax4.set_xlabel("Threshold")
ax4.set_ylabel("Alert Rate (%)")
ax4.set_title("Alert Volume vs Threshold")
ax4.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
# Workload planning
print("WORKLOAD SCENARIO ANALYSIS")
print("(Assuming 100,000 events/day, 50 alerts/analyst capacity)")
print("=" * 70)

scenarios = analyze_workload_scenarios(
    analyzer,
    daily_events=100000,
    analyst_capacity=50
)

print(scenarios.to_string(index=False))

In [ ]:
print(generate_calibration_report(analyzer, daily_events=100000))

---
# Summary

This notebook demonstrated the progression from simple statistical methods to ML classification:

1. **Baseline Modeling**: Establish "normal" before detecting "abnormal"
2. **Hypothesis Testing**: Formalize detection as statistical inference with p-values
3. **Time Series Preparation**: Respect temporal structure; never use random splits
4. **CUSUM Detection**: Detect persistent shifts with minimal false positives
5. **SVM Classification**: Supervised learning with careful hyperparameter tuning
6. **Threshold Calibration**: Balance detection rate against analyst workload

## Key Takeaways for Practitioners

- **Start simple**: Baselines and z-scores catch many anomalies
- **Quantify confidence**: p-values are more actionable than binary alerts
- **Respect time**: Use temporal splits, not random splits
- **Tune for operations**: Optimal F1 may not be optimal for your SOC
- **Document everything**: Reproducibility is essential for security research